# Notebook 18: Evaluation Metrics for NLP Models

## 📚 Historical Context

**Timeline**: 1966-present - From simple to sophisticated metrics

**The Evaluation Problem (Pre-2000s)**:
- **Early NLP**: Only accuracy for classification
- **Machine Translation**: Human evaluation only (slow, expensive)
- **Text Generation**: No automatic metrics
- **Problem**: Can't quickly iterate or compare models

**Key Innovations**:

**1977: Perplexity for Language Models**
- **Information theory** foundation
- **Idea**: How "surprised" is the model by test data?
- **Lower is better**: Less surprise = better prediction
- **Impact**: Standard LM metric for 40+ years

**2002: BLEU Score (Papineni et al.)**
- **Problem**: Need automatic translation evaluation
- **Idea**: Compare n-gram overlap with reference
- **Breakthrough**: Correlates with human judgment
- **Impact**: Enabled MT research explosion

**2004: ROUGE Scores (Lin)**
- **For summarization**: BLEU wasn't enough
- **Focus**: Recall (covering reference content)
- **Variants**: ROUGE-N, ROUGE-L, ROUGE-S
- **Impact**: Standard summarization metric

**2016: Squad and Exact Match (Rajpurkar et al.)**
- **Question answering**: Need strict correctness
- **Metrics**: Exact Match (EM) + F1
- **Dataset**: 100K+ QA pairs
- **Impact**: Drove QA research progress

**2018-Present: Task-Specific Metrics**
- **GLUE benchmark**: 9 tasks, 9 metrics
- **SuperGLUE**: Harder tasks, more metrics
- **BERTScore**: Contextual embedding similarity
- **Human evaluation**: Still the gold standard

**Modern Challenges (2020+)**:
- **Open-ended generation**: BLEU/ROUGE insufficient
- **Instruction following**: Need human preference data
- **ChatGPT era**: Metrics lag behind capabilities
- **Solution**: Multi-faceted evaluation + human studies

## 🎯 What You'll Learn

1. **Perplexity** - Language model evaluation
2. **BLEU & ROUGE** - Generation quality metrics
3. **F1, Precision, Recall** - Classification metrics
4. **Exact Match** - Question answering
5. **Human Evaluation** - Gold standard approaches
6. **Comprehensive Reports** - Putting it all together

## 💡 Why This Matters

**Choosing the right metric is critical**:
- **Wrong metric**: Optimize for the wrong thing
- **Right metric**: Measures what users care about
- **Multiple metrics**: Get complete picture

**Real-world impact**:
```
Example: Machine Translation
- Only BLEU: May produce unnatural text
- BLEU + human eval: Catches fluency issues
- BLEU + ROUGE + human: Complete picture
```

---

In [ ]:
# Import libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
import re
from typing import List, Dict, Tuple
import pandas as pd

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Evaluation metrics library ready!")

## Part 1: Perplexity for Language Models

### What is Perplexity?

**Intuition**: How "perplexed" (surprised) is the model by the test data?

**Mathematical Definition**:
```
Given sequence: w₁, w₂, ..., wₙ

Cross-entropy loss:
H = -(1/n) Σ log P(wᵢ | w₁...wᵢ₋₁)

Perplexity:
PPL = exp(H) = exp(-(1/n) Σ log P(wᵢ | w₁...wᵢ₋₁))
```

**Interpretation**:
```
PPL = 1:     Perfect prediction (never happens)
PPL = 10:    Model choosing from ~10 words on average
PPL = 100:   Model choosing from ~100 words on average
PPL = 1000:  Very uncertain model
```

**Why exponentiate?**
```
Cross-entropy is in log space (hard to interpret)
Perplexity is in probability space (interpretable as "branching factor")

Example:
If PPL = 50, model is as confused as if it had to uniformly 
choose from 50 words at each step
```

### Typical Perplexity Values:

```
Task                     Good PPL    State-of-art
Penn Treebank (word):    100-120     47.7 (2021)
WikiText-2 (word):       80-100      16.4 (2023)
WikiText-103 (word):     30-40       15.2 (2023)
1 Billion Word:          50-60       23.7 (2020)
OpenWebText (GPT-2):     ~30         ~20 (GPT-3)
```

### Important Notes:

**1. Vocabulary matters**:
```
Small vocab (10K):  Lower perplexity
Large vocab (50K):  Higher perplexity
BPE tokens:         In between

→ Only compare models with same tokenization!
```

**2. Perplexity doesn't measure**:
```
✗ Text quality
✗ Coherence
✗ Factual correctness
✓ Next-token prediction ability
```

**3. Lower is better**:
```
Lower PPL = Better prediction = Better language model
(for the training distribution)
```

In [ ]:
# Implement perplexity calculation
print("=" * 60)
print("PERPLEXITY FOR LANGUAGE MODELS")
print("=" * 60)

def calculate_perplexity(model, data_loader, device='cpu'):
    """
    Calculate perplexity of a language model.
    
    Args:
        model: Language model (outputs logits)
        data_loader: DataLoader with (input_ids, target_ids)
        device: 'cpu' or 'cuda'
    
    Returns:
        perplexity: float
    """
    model.eval()
    total_loss = 0
    total_tokens = 0
    
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            # Forward pass
            logits = model(inputs)
            
            # Calculate cross-entropy loss
            # Flatten logits and targets
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                reduction='sum'
            )
            
            total_loss += loss.item()
            total_tokens += targets.numel()
    
    # Average loss (cross-entropy)
    avg_loss = total_loss / total_tokens
    
    # Perplexity = exp(cross-entropy)
    perplexity = np.exp(avg_loss)
    
    return perplexity, avg_loss

# Simple language model for demonstration
class SimpleLM(nn.Module):
    """Simple LSTM language model."""
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.lstm(x)
        x = self.fc(x)
        return x

# Create synthetic data
vocab_size = 1000
seq_len = 20
n_samples = 100

# Generate random sequences (in practice, these would be real text)
torch.manual_seed(42)
inputs = torch.randint(0, vocab_size, (n_samples, seq_len))
targets = torch.randint(0, vocab_size, (n_samples, seq_len))

from torch.utils.data import TensorDataset, DataLoader
dataset = TensorDataset(inputs, targets)
dataloader = DataLoader(dataset, batch_size=16)

# Create models with different quality
print("\nComparing models with different perplexities:")
print("-" * 60)

# Model 1: Random (worst possible)
random_model = SimpleLM(vocab_size)
ppl1, loss1 = calculate_perplexity(random_model, dataloader)
print(f"Random model (untrained):")
print(f"  Cross-entropy: {loss1:.4f}")
print(f"  Perplexity:    {ppl1:.2f}")
print(f"  Interpretation: Choosing uniformly from ~{ppl1:.0f} words")

# Model 2: Partially trained
partial_model = SimpleLM(vocab_size)
optimizer = torch.optim.Adam(partial_model.parameters())
criterion = nn.CrossEntropyLoss()

# Train for a few epochs
partial_model.train()
for epoch in range(5):
    for batch_inputs, batch_targets in dataloader:
        optimizer.zero_grad()
        outputs = partial_model(batch_inputs)
        loss = criterion(outputs.view(-1, vocab_size), batch_targets.view(-1))
        loss.backward()
        optimizer.step()

ppl2, loss2 = calculate_perplexity(partial_model, dataloader)
print(f"\nPartially trained model:")
print(f"  Cross-entropy: {loss2:.4f}")
print(f"  Perplexity:    {ppl2:.2f}")
print(f"  Improvement:   {ppl1/ppl2:.2f}x better")

# Visualize relationship between loss and perplexity
print("\n" + "=" * 60)
print("RELATIONSHIP: LOSS vs PERPLEXITY")
print("=" * 60)

losses = np.linspace(0, 7, 100)
perplexities = np.exp(losses)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Loss vs Perplexity
axes[0].plot(losses, perplexities, linewidth=2)
axes[0].axhline(y=vocab_size, color='r', linestyle='--', label=f'Random (vocab size={vocab_size})')
axes[0].set_xlabel('Cross-Entropy Loss')
axes[0].set_ylabel('Perplexity')
axes[0].set_title('Perplexity = exp(Loss)')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Plot 2: Log scale
axes[1].semilogy(losses, perplexities, linewidth=2)
axes[1].axhline(y=vocab_size, color='r', linestyle='--', label=f'Random (vocab size={vocab_size})')
axes[1].set_xlabel('Cross-Entropy Loss')
axes[1].set_ylabel('Perplexity (log scale)')
axes[1].set_title('Log Scale View')
axes[1].grid(True, alpha=0.3, which='both')
axes[1].legend()

plt.tight_layout()
plt.show()

# Create reference table
print("\n" + "=" * 60)
print("PERPLEXITY REFERENCE TABLE")
print("=" * 60)

reference_data = [
    ["Random (1000 vocab)", 6.91, 1000],
    ["Poor model", 5.30, 200],
    ["Mediocre model", 4.61, 100],
    ["Good model", 3.91, 50],
    ["Great model", 3.00, 20],
    ["Excellent model", 2.30, 10],
]

print(f"\n{'Model Quality':<20} {'Loss':<10} {'Perplexity':<15} {'Interpretation'}")
print("-" * 70)
for name, loss, ppl in reference_data:
    print(f"{name:<20} {loss:<10.2f} {ppl:<15.1f} ~{ppl:.0f}-way choice")

print("\n" + "=" * 60)
print("KEY POINTS")
print("=" * 60)
print("\n1. Perplexity = exp(cross-entropy loss)")
print("2. Lower is better (less surprise)")
print("3. Interpretable as branching factor")
print("4. Only compare models with same tokenization")
print("5. Doesn't measure text quality directly")
print("=" * 60)

## Part 2: BLEU and ROUGE for Text Generation

### BLEU Score (Bilingual Evaluation Understudy)

**Purpose**: Machine translation evaluation

**Core Idea**: Compare n-gram overlap between candidate and reference(s)

**Formula**:
```
BLEU = BP × exp(Σ wₙ × log(pₙ))

Where:
pₙ = Precision for n-grams (n=1,2,3,4)
wₙ = Weights (typically uniform: 0.25 each)
BP = Brevity penalty (penalize short translations)
```

**N-gram Precision**:
```
p₁ = # matching unigrams / # candidate unigrams
p₂ = # matching bigrams / # candidate bigrams
p₃ = # matching trigrams / # candidate trigrams
p₄ = # matching 4-grams / # candidate 4-grams
```

**Brevity Penalty**:
```
BP = 1  if candidate length ≥ reference length
BP = exp(1 - reference_len/candidate_len) otherwise

Prevents gaming the system with very short translations
```

**Example**:
```
Reference:  "The cat sat on the mat"
Candidate:  "The cat on the mat"

Unigrams: 5/5 match → p₁ = 1.0
Bigrams:  3/4 match ("the cat", "on the", "the mat") → p₂ = 0.75
Trigrams: 1/3 match ("on the mat") → p₃ = 0.33
4-grams:  0/2 match → p₄ = 0.0

BLEU = BP × exp(0.25×(log 1.0 + log 0.75 + log 0.33 + log 0.001))
     ≈ 0.0 (because p₄ is ~0)
```

**BLEU Scores**:
```
0-10:   Very poor (mostly wrong)
10-20:  Poor (understandable but many errors)
20-30:  Acceptable (useful but imperfect)
30-40:  Good (few errors)
40-50:  Very good (high quality)
50-60:  Excellent (near human)
60+:    Usually indicates test set overlap
```

### ROUGE Scores (Recall-Oriented Understudy for Gisting Evaluation)

**Purpose**: Summarization evaluation (focuses on recall)

**Variants**:

**1. ROUGE-N**: N-gram overlap (like BLEU but recall-focused)
```
ROUGE-1 = # matching unigrams / # reference unigrams
ROUGE-2 = # matching bigrams / # reference bigrams
```

**2. ROUGE-L**: Longest Common Subsequence
```
Measures longest sequence (not necessarily consecutive)
Better captures sentence-level similarity
```

**3. ROUGE-S**: Skip-bigram
```
Allows gaps between words
More flexible matching
```

**Key Difference from BLEU**:
```
BLEU:  Precision-focused (don't add wrong words)
ROUGE: Recall-focused (don't miss important words)

Translation: Precision matters more (don't make stuff up)
Summarization: Recall matters more (cover key points)
```

**ROUGE Scores**:
```
ROUGE-1: 0.3-0.4 (typical summarization)
ROUGE-2: 0.1-0.2 (typical summarization)
ROUGE-L: 0.25-0.35 (typical summarization)
```

In [ ]:
# Implement BLEU and ROUGE from scratch
print("=" * 60)
print("BLEU AND ROUGE SCORES")
print("=" * 60)

from collections import Counter
import math

def get_ngrams(tokens: List[str], n: int) -> Counter:
    """Extract n-grams from token list."""
    ngrams = []
    for i in range(len(tokens) - n + 1):
        ngrams.append(tuple(tokens[i:i+n]))
    return Counter(ngrams)

def calculate_bleu(
    candidate: str,
    references: List[str],
    max_n: int = 4
) -> Dict[str, float]:
    """
    Calculate BLEU score.
    
    Args:
        candidate: Generated text
        references: List of reference translations
        max_n: Maximum n-gram size (default 4 for BLEU-4)
    
    Returns:
        Dictionary with BLEU score and components
    """
    # Tokenize
    cand_tokens = candidate.lower().split()
    ref_tokens_list = [ref.lower().split() for ref in references]
    
    # Calculate precision for each n
    precisions = []
    
    for n in range(1, max_n + 1):
        cand_ngrams = get_ngrams(cand_tokens, n)
        
        # Maximum count from all references
        max_ref_counts = Counter()
        for ref_tokens in ref_tokens_list:
            ref_ngrams = get_ngrams(ref_tokens, n)
            for ngram in cand_ngrams:
                max_ref_counts[ngram] = max(max_ref_counts[ngram], ref_ngrams[ngram])
        
        # Clipped counts
        clipped_counts = {
            ngram: min(count, max_ref_counts[ngram])
            for ngram, count in cand_ngrams.items()
        }
        
        # Precision
        numerator = sum(clipped_counts.values())
        denominator = sum(cand_ngrams.values())
        
        if denominator == 0:
            precision = 0
        else:
            precision = numerator / denominator
        
        precisions.append(precision)
    
    # Brevity penalty
    cand_len = len(cand_tokens)
    ref_len = min(len(ref_tokens) for ref_tokens in ref_tokens_list)
    
    if cand_len >= ref_len:
        bp = 1.0
    else:
        bp = math.exp(1 - ref_len / cand_len)
    
    # BLEU score
    if min(precisions) > 0:
        log_precisions = [math.log(p) for p in precisions]
        bleu = bp * math.exp(sum(log_precisions) / len(precisions))
    else:
        bleu = 0.0
    
    return {
        'bleu': bleu,
        'precisions': precisions,
        'bp': bp,
        'cand_len': cand_len,
        'ref_len': ref_len
    }

def calculate_rouge_n(
    candidate: str,
    reference: str,
    n: int = 1
) -> Dict[str, float]:
    """
    Calculate ROUGE-N score.
    
    Returns precision, recall, and F1.
    """
    cand_tokens = candidate.lower().split()
    ref_tokens = reference.lower().split()
    
    cand_ngrams = get_ngrams(cand_tokens, n)
    ref_ngrams = get_ngrams(ref_tokens, n)
    
    # Overlap
    overlap = sum((cand_ngrams & ref_ngrams).values())
    
    # Precision and recall
    precision = overlap / sum(cand_ngrams.values()) if cand_ngrams else 0
    recall = overlap / sum(ref_ngrams.values()) if ref_ngrams else 0
    
    # F1
    if precision + recall > 0:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = 0
    
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# Test examples
print("\nExample 1: Good Translation")
print("-" * 60)
reference = "The cat sat on the mat"
candidate = "The cat is sitting on the mat"

bleu_result = calculate_bleu(candidate, [reference])
rouge1 = calculate_rouge_n(candidate, reference, n=1)
rouge2 = calculate_rouge_n(candidate, reference, n=2)

print(f"Reference:  {reference}")
print(f"Candidate:  {candidate}")
print(f"\nBLEU Score: {bleu_result['bleu']:.4f}")
print(f"  Unigram precision:  {bleu_result['precisions'][0]:.4f}")
print(f"  Bigram precision:   {bleu_result['precisions'][1]:.4f}")
print(f"  Trigram precision:  {bleu_result['precisions'][2]:.4f}")
print(f"  4-gram precision:   {bleu_result['precisions'][3]:.4f}")
print(f"  Brevity penalty:    {bleu_result['bp']:.4f}")
print(f"\nROUGE-1 F1: {rouge1['f1']:.4f} (P={rouge1['precision']:.4f}, R={rouge1['recall']:.4f})")
print(f"ROUGE-2 F1: {rouge2['f1']:.4f} (P={rouge2['precision']:.4f}, R={rouge2['recall']:.4f})")

print("\n" + "="*60)
print("Example 2: Poor Translation")
print("-" * 60)
reference = "The cat sat on the mat"
candidate = "A dog was running in the park"

bleu_result = calculate_bleu(candidate, [reference])
rouge1 = calculate_rouge_n(candidate, reference, n=1)

print(f"Reference:  {reference}")
print(f"Candidate:  {candidate}")
print(f"\nBLEU Score: {bleu_result['bleu']:.4f}")
print(f"ROUGE-1 F1: {rouge1['f1']:.4f}")

print("\n" + "="*60)
print("Example 3: Perfect Match")
print("-" * 60)
reference = "The cat sat on the mat"
candidate = "The cat sat on the mat"

bleu_result = calculate_bleu(candidate, [reference])
rouge1 = calculate_rouge_n(candidate, reference, n=1)

print(f"Reference:  {reference}")
print(f"Candidate:  {candidate}")
print(f"\nBLEU Score: {bleu_result['bleu']:.4f}")
print(f"ROUGE-1 F1: {rouge1['f1']:.4f}")

# Visualize BLEU vs ROUGE
print("\n" + "="*60)
print("BLEU vs ROUGE COMPARISON")
print("="*60)

test_cases = [
    ("The cat sat on the mat", "The cat sat on the mat", "Perfect"),
    ("The cat sat on the mat", "The cat on the mat", "Missing word"),
    ("The cat sat on the mat", "The feline sat on the mat", "Synonym"),
    ("The cat sat on the mat", "On the mat sat the cat", "Reordered"),
    ("The cat sat on the mat", "The cat", "Too short"),
    ("The cat sat on the mat", "The cat sat on the mat and then went to sleep", "Too long"),
]

results = []
for ref, cand, desc in test_cases:
    bleu = calculate_bleu(cand, [ref])['bleu']
    rouge1 = calculate_rouge_n(cand, ref, n=1)['f1']
    rouge2 = calculate_rouge_n(cand, ref, n=2)['f1']
    results.append([desc, bleu, rouge1, rouge2])

# Plot
df = pd.DataFrame(results, columns=['Case', 'BLEU', 'ROUGE-1', 'ROUGE-2'])

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(df))
width = 0.25

ax.bar(x - width, df['BLEU'], width, label='BLEU', alpha=0.8)
ax.bar(x, df['ROUGE-1'], width, label='ROUGE-1', alpha=0.8)
ax.bar(x + width, df['ROUGE-2'], width, label='ROUGE-2', alpha=0.8)

ax.set_ylabel('Score')
ax.set_title('BLEU vs ROUGE Scores Across Different Cases')
ax.set_xticks(x)
ax.set_xticklabels(df['Case'], rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("KEY OBSERVATIONS")
print("="*60)
print("\n1. BLEU penalizes reordering more than ROUGE")
print("2. BLEU heavily penalizes length differences")
print("3. ROUGE-1 is more lenient (word-level matching)")
print("4. ROUGE-2 is stricter (requires bigram matches)")
print("5. Neither captures semantic similarity well")
print("="*60)

## Part 3: Classification Metrics - F1, Precision, Recall

### Confusion Matrix Foundation:

```
                 Predicted
              Positive  Negative
Actual Pos    TP        FN
      Neg    FP        TN

TP: True Positive (correctly predicted positive)
FP: False Positive (incorrectly predicted positive)
FN: False Negative (incorrectly predicted negative)
TN: True Negative (correctly predicted negative)
```

### Key Metrics:

**Accuracy**:
```
Accuracy = (TP + TN) / (TP + FP + FN + TN)

Problem: Misleading with imbalanced classes
Example: 95% negative class → always predict negative = 95% accuracy!
```

**Precision** (How many predicted positives are correct?):
```
Precision = TP / (TP + FP)

High precision: Few false alarms
Use when: False positives are costly
Example: Spam detection (don't mark real emails as spam)
```

**Recall** (How many actual positives did we find?):
```
Recall = TP / (TP + FN)

High recall: Few missed positives
Use when: False negatives are costly
Example: Disease detection (don't miss sick patients)
```

**F1 Score** (Harmonic mean of precision and recall):
```
F1 = 2 × (Precision × Recall) / (Precision + Recall)

Balanced metric
Use when: Want balance between precision and recall
Most common in NLP competitions
```

### Multi-Class Metrics:

**Macro-average**:
```
Calculate metric for each class
Take unweighted average

Treats all classes equally
Good for imbalanced datasets
```

**Micro-average**:
```
Pool all TPs, FPs, FNs
Calculate one metric

Weighted by class frequency
Similar to accuracy for balanced datasets
```

**Weighted-average**:
```
Calculate metric for each class
Weight by class support

Accounts for class imbalance
Most common in practice
```

In [ ]:
# Implement classification metrics
print("=" * 60)
print("CLASSIFICATION METRICS")
print("=" * 60)

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

def calculate_metrics(y_true, y_pred, labels=None):
    """
    Calculate comprehensive classification metrics.
    """
    # Basic metrics
    accuracy = accuracy_score(y_true, y_pred)
    
    # Precision, recall, F1
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, labels=labels, zero_division=0
    )
    
    # Macro average
    macro_precision = precision.mean()
    macro_recall = recall.mean()
    macro_f1 = f1.mean()
    
    # Micro average
    micro_precision, micro_recall, micro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='micro', zero_division=0
    )
    
    # Weighted average
    weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )
    
    return {
        'accuracy': accuracy,
        'per_class': {
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'support': support
        },
        'macro': {
            'precision': macro_precision,
            'recall': macro_recall,
            'f1': macro_f1
        },
        'micro': {
            'precision': micro_precision,
            'recall': micro_recall,
            'f1': micro_f1
        },
        'weighted': {
            'precision': weighted_precision,
            'recall': weighted_recall,
            'f1': weighted_f1
        }
    }

# Example 1: Balanced dataset
print("\nExample 1: Balanced Dataset")
print("-" * 60)
y_true = [0, 0, 1, 1, 2, 2] * 10  # 30 samples each class
y_pred = [0, 0, 1, 1, 2, 2] * 8 + [0, 1, 1, 0, 2, 1] * 2  # Some errors

metrics = calculate_metrics(y_true, y_pred, labels=[0, 1, 2])

print(f"Accuracy: {metrics['accuracy']:.4f}")
print(f"\nPer-class metrics:")
for i in range(3):
    print(f"  Class {i}:")
    print(f"    Precision: {metrics['per_class']['precision'][i]:.4f}")
    print(f"    Recall:    {metrics['per_class']['recall'][i]:.4f}")
    print(f"    F1:        {metrics['per_class']['f1'][i]:.4f}")
    print(f"    Support:   {metrics['per_class']['support'][i]}")

print(f"\nMacro-average:")
print(f"  Precision: {metrics['macro']['precision']:.4f}")
print(f"  Recall:    {metrics['macro']['recall']:.4f}")
print(f"  F1:        {metrics['macro']['f1']:.4f}")

# Example 2: Imbalanced dataset
print("\n" + "="*60)
print("Example 2: Imbalanced Dataset (90% class 0)")
print("-" * 60)
y_true = [0] * 90 + [1] * 5 + [2] * 5
y_pred = [0] * 88 + [1] * 2 + [1] * 4 + [0] * 1 + [2] * 3 + [1] * 2

metrics = calculate_metrics(y_true, y_pred, labels=[0, 1, 2])

print(f"Accuracy: {metrics['accuracy']:.4f} ← Misleading!")
print(f"\nMacro-average (treats all classes equally):")
print(f"  F1: {metrics['macro']['f1']:.4f}")
print(f"\nWeighted-average (accounts for imbalance):")
print(f"  F1: {metrics['weighted']['f1']:.4f}")

# Visualize confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
           xticklabels=[0, 1, 2], yticklabels=[0, 1, 2])
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')
axes[0].set_title('Confusion Matrix (Counts)')

# Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=axes[1],
           xticklabels=[0, 1, 2], yticklabels=[0, 1, 2])
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')
axes[1].set_title('Confusion Matrix (Normalized)')

plt.tight_layout()
plt.show()

# Show when to use each metric
print("\n" + "="*60)
print("WHEN TO USE WHICH METRIC")
print("="*60)

use_cases = [
    ["Accuracy", "Balanced classes only", "General overview"],
    ["Precision", "False positives costly", "Spam detection"],
    ["Recall", "False negatives costly", "Disease detection"],
    ["F1", "Balance needed", "Most NLP tasks"],
    ["Macro F1", "All classes equal", "Imbalanced classes"],
    ["Weighted F1", "Account for frequency", "Reporting"],
]

print(f"\n{'Metric':<15} {'Use When':<30} {'Example'}")
print("-" * 70)
for metric, use, example in use_cases:
    print(f"{metric:<15} {use:<30} {example}")

print("="*60)

## Part 4: Exact Match and F1 for Question Answering

### SQuAD-Style Metrics:

**Exact Match (EM)**:
```python
def exact_match(prediction, ground_truth):
    # Normalize both strings
    pred = normalize(prediction)
    gt = normalize(ground_truth)
    
    return 1.0 if pred == gt else 0.0
```

**Normalization** (SQuAD standard):
```python
1. Lowercase
2. Remove articles (a, an, the)
3. Remove punctuation
4. Remove extra whitespace
```

**Token-level F1**:
```python
# Tokenize both strings
pred_tokens = set(normalize(prediction).split())
gt_tokens = set(normalize(ground_truth).split())

# Calculate overlap
overlap = pred_tokens & gt_tokens

# Precision and recall
precision = len(overlap) / len(pred_tokens)
recall = len(overlap) / len(gt_tokens)

# F1
f1 = 2 * precision * recall / (precision + recall)
```

### Why Both Metrics?

```
Question: "Who is the president of France?"
Ground truth: "Emmanuel Macron"

Prediction 1: "Emmanuel Macron"
  EM: 1.0 ✓
  F1: 1.0 ✓

Prediction 2: "Macron"
  EM: 0.0 ✗ (not exact match)
  F1: 0.67 ✓ (partial credit)

Prediction 3: "The president is Emmanuel Macron"
  EM: 0.0 ✗ (extra words)
  F1: 1.0 ✓ (contains all tokens)
```

**EM**: Strict, no partial credit
**F1**: Lenient, gives partial credit

### Typical Scores:

```
SQuAD 1.1:
  Human:     EM=82.3%, F1=91.2%
  BERT-base: EM=81.2%, F1=88.5%
  BERT-large: EM=84.1%, F1=90.9%

SQuAD 2.0 (with unanswerable):
  Human:     EM=86.8%, F1=89.5%
  BERT-large: EM=81.9%, F1=84.5%
```

In [ ]:
# Implement QA metrics
print("=" * 60)
print("QUESTION ANSWERING METRICS")
print("=" * 60)

import re
import string

def normalize_answer(s):
    """Normalize answer string (SQuAD style)."""
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)
    
    def white_space_fix(text):
        return ' '.join(text.split())
    
    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)
    
    def lower(text):
        return text.lower()
    
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def exact_match_score(prediction, ground_truth):
    """Calculate exact match score."""
    return 1.0 if normalize_answer(prediction) == normalize_answer(ground_truth) else 0.0

def f1_score_qa(prediction, ground_truth):
    """Calculate token-level F1 score."""
    pred_tokens = normalize_answer(prediction).split()
    gt_tokens = normalize_answer(ground_truth).split()
    
    if len(pred_tokens) == 0 or len(gt_tokens) == 0:
        return int(pred_tokens == gt_tokens)
    
    common = Counter(pred_tokens) & Counter(gt_tokens)
    num_same = sum(common.values())
    
    if num_same == 0:
        return 0.0
    
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gt_tokens)
    f1 = 2 * precision * recall / (precision + recall)
    
    return f1

def evaluate_qa(predictions, ground_truths):
    """Evaluate QA predictions."""
    em_scores = []
    f1_scores = []
    
    for pred, gts in zip(predictions, ground_truths):
        # Take max over all ground truths (multiple acceptable answers)
        em = max(exact_match_score(pred, gt) for gt in gts)
        f1 = max(f1_score_qa(pred, gt) for gt in gts)
        
        em_scores.append(em)
        f1_scores.append(f1)
    
    return {
        'exact_match': np.mean(em_scores),
        'f1': np.mean(f1_scores),
        'em_scores': em_scores,
        'f1_scores': f1_scores
    }

# Test examples
print("\nTest Cases:")
print("-" * 60)

test_cases = [
    ("Emmanuel Macron", ["Emmanuel Macron"], "Perfect match"),
    ("Macron", ["Emmanuel Macron"], "Partial match"),
    ("The president is Emmanuel Macron", ["Emmanuel Macron"], "Extra words"),
    ("Paris", ["Paris", "paris", "PARIS"], "Case variations"),
    ("1984", ["Nineteen Eighty-Four", "1984"], "Multiple references"),
    ("The answer is 42", ["42"], "With article"),
]

for pred, gts, desc in test_cases:
    em = max(exact_match_score(pred, gt) for gt in gts)
    f1 = max(f1_score_qa(pred, gt) for gt in gts)
    
    print(f"\n{desc}:")
    print(f"  Prediction: '{pred}'")
    print(f"  Reference:  {gts}")
    print(f"  EM: {em:.2f}  F1: {f1:.4f}")

# Simulate QA system evaluation
print("\n" + "="*60)
print("QA SYSTEM EVALUATION")
print("="*60)

# Simulated predictions and ground truths
predictions = [
    "Paris",
    "The Eiffel Tower",
    "1889",
    "324 meters",
    "Gustave Eiffel",
    "Iron",
    "7 million",
    "Champ de Mars",
    "French",
    "Two years"
]

ground_truths = [
    ["Paris", "paris"],
    ["Eiffel Tower", "the Eiffel Tower"],
    ["1889"],
    ["324 meters", "324m"],
    ["Gustave Eiffel"],
    ["Iron", "iron"],
    ["7 million", "seven million"],
    ["Champ de Mars"],
    ["French", "France"],
    ["2 years", "two years"]
]

results = evaluate_qa(predictions, ground_truths)

print(f"\nOverall Metrics:")
print(f"  Exact Match: {results['exact_match']*100:.2f}%")
print(f"  F1 Score:    {results['f1']*100:.2f}%")

# Per-question breakdown
print(f"\nPer-Question Breakdown:")
print(f"{'Prediction':<20} {'Reference':<20} {'EM':<6} {'F1':<8}")
print("-" * 60)
for i, (pred, gts) in enumerate(zip(predictions, ground_truths)):
    em = results['em_scores'][i]
    f1 = results['f1_scores'][i]
    ref = gts[0] if len(gts) == 1 else f"{gts[0]} (+{len(gts)-1})"
    print(f"{pred:<20} {ref:<20} {em:<6.0f} {f1:<8.4f}")

print("\n" + "="*60)
print("KEY POINTS")
print("="*60)
print("\n1. EM is strict - no partial credit")
print("2. F1 gives partial credit for token overlap")
print("3. Normalization is critical (lowercase, remove articles, etc.)")
print("4. Multiple references help handle variation")
print("5. Both metrics together give complete picture")
print("="*60)

## 📊 Summary

### Metric Selection Guide:

| Task | Primary Metrics | Why |
|------|----------------|-----|
| Language Modeling | Perplexity | Measures prediction quality |
| Machine Translation | BLEU, human eval | Precision + fluency |
| Summarization | ROUGE-1, ROUGE-2, ROUGE-L | Recall + coverage |
| Classification (balanced) | Accuracy, F1 | Overall performance |
| Classification (imbalanced) | Macro F1, per-class F1 | Fair to all classes |
| Question Answering | Exact Match, F1 | Strict + lenient |
| Named Entity Recognition | Token-level F1 | Entity boundaries |
| Sentiment Analysis | Accuracy, Macro F1 | Class balance varies |

### Best Practices:

**1. Report multiple metrics**:
```python
# Don't just report one number
print(f"Accuracy: {acc}")

# Report comprehensive metrics
print(f"Accuracy:  {acc:.4f}")
print(f"Macro F1:  {macro_f1:.4f}")
print(f"Per-class F1: {per_class_f1}")
```

**2. Use appropriate averaging**:
```python
# Balanced: micro or weighted
# Imbalanced: macro (treats classes equally)
```

**3. Include human evaluation**:
```python
# Automatic metrics don't capture everything
# For generation: sample and manually review
# For critical applications: dedicated human eval
```

**4. Compare to baselines**:
```python
# Always compare to:
# - Random baseline
# - Majority class baseline
# - Previous state-of-the-art
```

### Next Notebook:

**19_curriculum_learning.ipynb**

We'll explore:
- Easy-to-hard training strategies
- Sorting by difficulty (length, complexity)
- Staged training implementation
- When curriculum learning helps
- Comparison with standard training
- Research paper examples and results

---

**Historical Note**: Evaluation metrics have evolved from simple accuracy (1960s) to sophisticated multi-faceted evaluation (2020s). The key insight: no single metric captures everything. Modern practice combines automatic metrics for quick iteration with human evaluation for final validation. The tension between automatic and human evaluation continues, especially with modern LLMs producing fluent but potentially incorrect text.